##Load Library dan Dataset

In [ ]:
!pip install datasets soundfile librosa -q

In [ ]:
import numpy as np
import librosa
import soundfile as sf
from io import BytesIO
from datasets import load_dataset, concatenate_datasets, Dataset, Audio

In [ ]:
def load_n_samples(name, label, n=400):
    stream = load_dataset("cgeorgiaw/animal-sounds", name, streaming=True)["train"]
    samples = []
    for i, example in enumerate(stream):
        if i >= n:
            break
        samples.append({
            "audio": example["audio"],
            "label": label
        })
    return Dataset.from_list(samples)

birds       = load_n_samples("birds",       "birds")
macaque     = load_n_samples("macaque",     "macaque")
giant_otter = load_n_samples("giant_otter", "giant_otter")
orca        = load_n_samples("orca",        "orca")
zebra_finch = load_n_samples("zebra_finch", "zebra_finch")

# Gabungkan
dataset = concatenate_datasets([birds, macaque, giant_otter, orca, zebra_finch])
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

In [ ]:
print(f"Total sampel: {len(dataset)}")
print(f"Kolom      : {dataset.column_names}")

##Preprocessing


###1. Resampling

In [ ]:
# Cast audio supaya bisa di-decode
dataset = dataset.cast_column("audio", Audio())

# Cek sample rate tiap kelas
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

In [ ]:
# Resample semua ke 22050 Hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

# Verifikasi
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    sr = dataset[i]["audio"]["sampling_rate"]
    print(f"{label:20s} → sample rate: {sr} Hz")

###2. Stereo ke Mono

In [ ]:
for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    print(f"{label:20s} → shape: {audio.shape}")

-> Semua audio sudah mono, tidak perlu konversi stereo ke mono.

###3. Trim Silence

In [ ]:
def trim_silence(example):
    audio = np.array(example["audio"]["array"], dtype=np.float32)
    audio_trimmed, _ = librosa.effects.trim(audio, top_db=20)

    # Simpan sebagai bytes supaya tidak error saat encode
    buffer = BytesIO()
    sf.write(buffer, audio_trimmed, 22050, format="wav")
    example["audio"] = {"bytes": buffer.getvalue(), "path": None}
    return example

dataset = dataset.map(trim_silence)

# Verifikasi
from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    duration = len(audio) / 22050
    print(f"{label:20s} → {len(audio)} sampel ({duration:.2f} detik)")

###4. Filter Durasi

In [ ]:
#Cek statistik
durations = []
for example in dataset:
    audio = np.array(example["audio"]["array"])
    duration = len(audio) / 22050
    durations.append(duration)

durations = np.array(durations)
print(f"Total sampel       : {len(durations)}")
print(f"Durasi min         : {durations.min():.2f} detik")
print(f"Durasi max         : {durations.max():.2f} detik")
print(f"Durasi rata-rata   : {durations.mean():.2f} detik")
print(f"Sampel > 3 detik   : {(durations > 3.0).sum()}")
print(f"Sampel <= 3 detik  : {(durations <= 3.0).sum()}")

In [ ]:
#Cek jumlah sampel yang lebih dari 3 detik
for i, label in zip(
    [(0,400), (400,800), (800,1200), (1200,1600), (1600,2000)],
    ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]
):
    start, end = i
    count = 0
    for j in range(start, end):
        audio = np.array(dataset[j]["audio"]["array"])
        duration = len(audio) / 22050
        if duration > 3.0:
            count += 1
    print(f"{label:20s} → {count}/400 sampel > 3 detik")

In [ ]:
#Cek jumlah sampel dengan threshold tertentu
thresholds = [0.5, 1.0, 1.5, 2.0]

for thresh in thresholds:
    print(f"\n--- Threshold > {thresh} detik ---")
    for i, label in zip(
        [(0,400), (400,800), (800,1200), (1200,1600), (1600,2000)],
        ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]
    ):
        start, end = i
        count = sum(
            1 for j in range(start, end)
            if len(np.array(dataset[j]["audio"]["array"])) / 22050 > thresh
        )
        print(f"  {label:20s} → {count}/400 sampel")

-> Filter durasi tidak dilakukan karena dengan Threshold 1.0 detik, ada kelas yang tidak memiliki sampel. Sementara jika diset ke 0.5 detik, filter kurang informatif.

###5. Fixing Duration (Padding/Truncation)

-> Pad/trim dilakukan ke 2 detik karena rata-rata durasi adalah 1.96 detik sehingga sampel yang lebih pendek tidak akan terlalu banyak padding-nya dan yang lebih panjang hanya dipotong sedikit.

In [ ]:
TARGET_SR      = 22050
TARGET_SAMPLES = TARGET_SR * 2  # 2 detik = 44100 sampel

import soundfile as sf
from io import BytesIO

def pad_trim(example):
    audio = np.array(example["audio"]["array"], dtype=np.float32)

    if len(audio) < TARGET_SAMPLES:
        # Pad dengan nol di akhir
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
    else:
        # Trim dari depan
        audio = audio[:TARGET_SAMPLES]

    buffer = BytesIO()
    sf.write(buffer, audio, TARGET_SR, format="wav")
    example["audio"] = {"bytes": buffer.getvalue(), "path": None}
    return example

dataset = dataset.map(pad_trim)

# Verifikasi
from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=22050))

for i, label in zip([0, 400, 800, 1200, 1600], ["birds", "macaque", "giant_otter", "orca", "zebra_finch"]):
    audio = np.array(dataset[i]["audio"]["array"])
    duration = len(audio) / 22050
    print(f"{label:20s} → {len(audio)} sampel ({duration:.2f} detik)")